# Model Comparison & Diff

Compare activations between two models to find where they diverge — useful for understanding what fine-tuning changed, or comparing model variants.

Here we compare **GPT-2** (12 layers) against **DistilGPT-2** (6 layers, distilled from GPT-2). They share the same module naming scheme (`transformer.h.<i>...`), so `interpkit.diff` automatically aligns the modules they have in common and reports per-module cosine distances.

In [1]:
import interpkit

model_a = interpkit.load("gpt2")           # 12 layers
model_b = interpkit.load("distilgpt2")     # 6 layers, distilled from gpt2

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Diff

Compare per-module activation distances (cosine distance) between the two models on the same input.

In [2]:
results = interpkit.diff(
    model_a,
    model_b,
    "The capital of France is",
)

diff: skipped 6 modules only in GPT2LMHeadModel. Comparing 6 shared modules.

Model Diff — GPT2LMHeadModel vs GPT2LMHeadModel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

transformer.h.1                      ████████████████████████    0.5783
transformer.h.5                      ███████████████████▌        0.4700
transformer.h.0                      █████                       0.1267
transformer.h.4                      ▌                           0.0139
transformer.h.3                                                  0.0066
transformer.h.2                                                  0.0041

╭─────────── Most changed ───────────╮
│  transformer.h.1  distance 0.5783  │
╰────────────────────────────────────╯

DistilGPT-2 was trained to mimic GPT-2's outputs but with half the layers and ~40% of the parameters. The resulting per-module cosine distances are real (non-zero) and tend to grow as you go deeper — early layers stay close to the teacher, later layers diverge more.

The modules with the largest cosine distance are the ones whose representations have drifted most from the teacher; those are usually the most interesting places to investigate further (e.g. with `trace`, `dla`, or `head_activations`).

## Export the diff chart

In [3]:
# results = interpkit.diff(model_a, model_b, "The capital of France is", save="diff.png")

## Side-by-side causal tracing

Run causal tracing on both models and compare which components are most important.
Use `mode="position"` for position-aware (Meng et al.) 2D heatmaps.

In [4]:
clean = "The capital of France is"
corrupted = "The capital of Poland is"

# Module-level tracing (default)
print("Model A — module trace:")
results_a = model_a.trace(clean, corrupted, top_k=5)

print("\nModel B — module trace:")
results_b = model_b.trace(clean, corrupted, top_k=5)

# Position-aware tracing (Meng et al. 2022) — 2D (layer x position) heatmap
print("\nModel A — position trace:")
pos_a = model_a.trace(clean, corrupted, mode="position")

Model A — module trace:


Scanning top 5 of 76 modules by proxy score.

Output()

Causal Trace — GPT2LMHeadModel  top 5 of 76 modules ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

transformer.h.10.attn.c_proj          attn     ████████████████████████    3.854
transformer.h.9.attn.c_proj           attn     ███████████████████▌        3.169
transformer.h.10.mlp.c_proj           mlp      █████████                   1.505
lm_head                               head     ██████                      1.000
transformer.ln_f                      norm     ██████                      1.000

...

Output()


Model A — position trace:


Position-Aware Causal Trace ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Rank   Layer              Position   Token     Effect  
 ─────────────────────────────────────────────────────── 
     1   transformer.h.9           3   ĠFrance    5.501  
     2   transformer.h.8           3   ĠFrance    4.871  
     3   transformer.h.7           3   ĠFrance    4.077  
     4   transformer.h.6           3   ĠFrance    3.623  
     5   transformer.h.5           3   ĠFrance    2.903  
     6   transformer.h.4           3   ĠFrance    2.034  
     7   transformer.h.2           3   ĠFrance    1.619  
     8   transformer.h.3           3   ĠFrance    1.495  
     9   transformer.h.1           3   ĠFrance    1.333  
    10   transformer.h.10          4   Ġis        1.296 

Heatmap: 12 layers × 5 positions. Use --save to export the full heatmap.

## Side-by-side logit lens

Compare how predictions evolve through layers in each model.

In [5]:
print("Model A:")
lens_a = model_a.lens("The capital of France is")

print("\nModel B:")
lens_b = model_b.lens("The capital of France is")

Model A:


Logit Lens — GPT2LMHeadModel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Layer              Top-1 Token    Prob                  Top-5 Tokens                                             
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  transformer.h.0     not          0.332   ███             not (0.33),  now (0.13),  still (0.08),  also (0.07),   
                                                          a (0.04)                                                 
  transformer.h.1     now          0.271   ██▌             now (0.27),  not (0.26),  also (0.16),  still (0.08),   
                                                          unlikely (0.02)                                          
  transformer.h.2     now          0.366   ███▌            now (0.37),  not (0.17),  also (0.15),  still (0.09),   
                                                          currently (0.03)                                         
  transformer.h.3     now          0.515   █████           now (0.51),  also (0.14),  not (0.13),  currently       
                                                          (0.04),  still (0.04)                                    
  transformer.h.4     now          0.627   ██████          now (0.63),  not (0.08),  also (0.05),  still (0.05),   
                                                          already (0.03)                                           
  transformer.h.5     now          0.552   █████▌          now (0.55),  still (0.11),  not (0.09),  also (0.07),   
                                                          already (0.03)                                           
  transformer.h.6     now          0.572   █████▌          now (0.57),  not (0.12),  still (0.06),  also (0.05),   
                                                          a (0.02)                                                 
  transformer.h.7     now          0.689   ██████▌         now (0.69),  also (0.06),  still (0.06),  not (0.05),   
                                                          already (0.03)                                           
  transformer.h.8     now          0.423   ████            now (0.42),  located (0.15),  also (0.07),  not         
                                                          (0.07),  still (0.05)                                    
  transformer.h.9     France       0.621   ██████          France (0.62),  Paris (0.18),  now (0.04),  located     
                                                          (0.03),  French (0.01)                                   
  transformer.h.10    France       0.242   ██              France (0.24),  Paris (0.14),  now (0.10),  the         
                                                          (0.04),  located (0.02)                                  
  transformer.h.11    the          0.085   ▌               the (0.08),  now (0.05),  a (0.05),  France (0.03),     
                                                          Paris (0.03)                                            


Model B:


Logit Lens — GPT2LMHeadModel ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

 Layer             Top-1 Token    Prob                  Top-5 Tokens                                              
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  transformer.h.0    not          0.681   ██████▌         not (0.68),  still (0.09),  also (0.05),  now (0.05),    
                                                         a (0.02)                                                  
  transformer.h.1    not          0.461   ████▌           not (0.46),  now (0.16),  also (0.11),  still (0.07),    
                                                         currently (0.04)                                          
  transformer.h.2    not          0.451   ████▌           not (0.45),  now (0.34),  also (0.05),  still (0.02),    
                                                         currently (0.02)                                          
  transformer.h.3    now          0.854   ████████▌       now (0.85),  currently (0.03),  located (0.03),  not     
                                                         (0.02),  still (0.01)                                     
  transformer.h.4    now          0.228   ██              now (0.23),  located (0.08),  known (0.06),  the         
                                                         (0.05),  currently (0.04)                                 
  transformer.h.5    the          0.118   █               the (0.12),  home (0.08),  now (0.05),  a (0.05),        
                                                         being (0.04)                                             

## Side-by-side DLA

Compare which components are most important for the prediction in each model.

In [6]:
print("Model A — DLA:")
dla_a = model_a.dla("The capital of France is", top_k=5)

print("\nModel B — DLA:")
dla_b = model_b.dla("The capital of France is", top_k=5)

Model A — DLA:


Direct Logit Attribution ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────╮
│  Target token: " the"  |  Total logit sum: 364.456  │
╰─────────────────────────────────────────────────────╯

 Component   Type   Contribution                               
 ────────────────────────────────────────────────────────────── 
  L11.attn    attn      +204.7007   ████████████████████████    
  L10.mlp     mlp        +67.3138   ███████▌                    
  L9.mlp      mlp        +27.4421   ███                         
  L8.mlp      mlp        +23.0129   ██▌                         
  L0.mlp      mlp        +15.5874   █▌                          
  ...                               (19 more)

Per-Head Breakdown  (top 5)

...

Direct Logit Attribution ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────╮
│  Target token: " the"  |  Total logit sum: 386.174  │
╰─────────────────────────────────────────────────────╯

 Component   Type   Contribution                               
 ────────────────────────────────────────────────────────────── 
  L5.attn     attn      +259.3835   ████████████████████████    
  L4.mlp      mlp        +53.2037   ████▌                       
  L5.mlp      mlp        +31.8267   ██▌                         
  L0.attn     attn       +15.8333   █                           
  L3.mlp      mlp        +10.3489   ▌                           
  ...                               (7 more)

Per-Head Breakdown  (top 5)

 Head     Contribution                               
 ──────────────────────────────────────────────────── 
  L5.H0       +126.8536   ████████████████████████    
  L5.H8        +85.2662   ████████████████            
  L5.H1         +4.8997   ▌                           
  L5.H11        +4.3628   ▌                           
  L5.H7         +4.3409   ▌                           
  L0.H11        -1.8924                               
  L1.H10        -1.9307                               
  L2.H7         -2.6992   ▌                           
  L4.H5         -2.8182   ▌                           
  L4.H8         -3.0933   ▌

## Comparing other model variants

`diff` aligns modules by name, so it works best when both models share the same naming scheme. Common pairs that work out of the box:

- **A base model vs your fine-tune** — surface exactly which modules the fine-tuning moved.
- **Same family, different size** — e.g. `gpt2` vs `gpt2-medium`. Shared module names (`transformer.h.0`...`transformer.h.11`) are compared; the extra layers in the larger model are skipped with a warning.
- **Distilled vs teacher** — what we just did.

```python
small = interpkit.load("gpt2")
medium = interpkit.load("gpt2-medium")

interpkit.diff(small, medium, "The capital of France is", save="size_comparison.png")
```